# C6 example 3/4: `WETensorProduct` with external circular harmonics

This same-node architecture contains exactly three 3D vectors and two scalars, packed as $3E_1\oplus5A$. Each vector $v_a$ constructs its own circular-harmonic filter $Y(v_a)$. One symmetric invariant summary of the three vectors produces a single reduced-weight vector $w$, the exact same $w$ is broadcast to all three filters, and their contributions are summed. There is no additional geometry vector.

`WETensorProduct` expects filter features to be computed externally, so this notebook performs

$$m=\sum_{a=1}^{3}\operatorname{WETP}\left(x,Y(v_a),w(I(v_1,v_2,v_3))\right),$$

where $I$ is the mean of the three pairs $(\lVert(v_a)_{xy}\rVert,(v_a)_z)$. It is C6-invariant and permutation-invariant over the filter-generating vectors. The default circular bandlimit is $L_{full}=3$.

In [ ]:
import torch
from we3nn import CircularHarmonics, CyclicGroup, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
harmonics = CircularHarmonics(G)  # max_frequency=None -> floor(6/2)=3
filter_rep = harmonics.rep_out
print('harmonic frequencies: 0..', harmonics.max_frequency)
print('filter representation:', filter_rep.name)
print('dimensions:', input_rep.size, 'x', filter_rep.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)

vector_angles = torch.atan2(vectors[..., 1], vectors[..., 0])
Y = harmonics(vector_angles)  # shape: [batch, 3 vector filters, harmonic components]
print('three vector angles:', vector_angles)
print('three external circular filters Y(v_a):', Y)

## Step 1: visualize the three filter-generating vectors

Every colored arrow is both a node feature and the source of one filter. The filtering machinery is shared; only the vector direction and its invariant magnitudes differ.

In [ ]:
import matplotlib.pyplot as plt

fig_input = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig_input.add_subplot(111, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'node feature v{index+1}')
limit = 1.15 * vectors.abs().max().item()
ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit), xlabel='x', ylabel='y', zlabel='z', title='Each node vector constructs one shared filter')
ax.set_box_aspect((1, 1, 1)); ax.legend(fontsize=8)
print('scalar node features:', scalars[0].tolist())
plt.show()

## Step 2: see the circular harmonics on their domain

A circular harmonic is a scalar component of a function on $S^1$. Each polar panel evaluates one component continuously around the unit circle. The radius is $1+0.35Y(\theta)/\max|Y|$: values outside the dashed unit circle are positive and values inside are negative. Circles, squares, and triangles mark the six C6 orientations of node vectors 1, 2, and 3 respectively.

In [ ]:
import math
import matplotlib.pyplot as plt

theta_grid = torch.linspace(0.0, 2.0 * math.pi, 361)
Y_circle = harmonics(theta_grid).detach()
orbit_angles = vector_angles[0, :, None] + torch.arange(6)[None, :] * (2.0 * math.pi / 6.0)
Y_orbit = harmonics(orbit_angles).detach()

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

fig_circle, axes = plt.subplots(2, 4, figsize=(13, 7), subplot_kw={'projection': 'polar'}, constrained_layout=True)
for component, (ax, label) in enumerate(zip(axes.flat, harmonic_labels)):
    scale = Y_circle[:, component].abs().max().clamp_min(1e-8)
    radius = 1.0 + 0.35 * Y_circle[:, component] / scale
    orbit_radius = 1.0 + 0.35 * Y_orbit[..., component] / scale
    ax.plot(theta_grid, torch.ones_like(theta_grid), color='gray', linestyle='--', linewidth=0.8)
    ax.plot(theta_grid, radius, linewidth=2)
    for vector_index, marker in enumerate(('o', 's', '^')):
        ax.scatter(orbit_angles[vector_index], orbit_radius[vector_index], c=torch.arange(6), cmap='hsv', marker=marker, s=30, zorder=3)
    ax.set_ylim(0.6, 1.4); ax.set_yticklabels([]); ax.set_title(label)
axes.flat[-1].set_axis_off()
fig_circle.suptitle('Every circular-harmonic component used by C6 (frequencies 0, 1, 2, 3)', fontsize=14)
plt.show()

## Step 3: build the externally filtered architecture

For each layer, the finite-group tensors $C_p$ are fixed, each $Y_j(v_a)$ carries its vector's angular dependence, and one symmetric invariant summary supplies a single coefficient vector $w_p(I)$ shared by all three filters:

$$z_o=\sum_{a=1}^{3}\sum_p w_p(I)(C_p)_{oij}x_iY_j(v_a).$$

The hidden regular representations admit coordinatewise `PointActiv`. Within a layer, the tensor product and radial network are shared across $a$; the input and output layers have separate parameters.

In [ ]:
class ExternalHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = harmonics
        self.input_layer = nn.WETensorProduct(
            input_rep, filter_rep, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.WETensorProduct(
            hidden_rep, filter_rep, output_rep, shared_weights=False
        )
        # One radial network is shared across all three vector filters.
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    @staticmethod
    def repeat_for_three_filters(features, representation):
        repeated = features.tensor.unsqueeze(-2).expand(
            *features.tensor.shape[:-1], 3, features.tensor.shape[-1]
        )
        return nn.RepresentationTensor(repeated, representation)

    def forward(self, features):
        # Recover the same three physical vectors already stored inside x.
        node_vectors, _ = unpack_input(features.tensor)
        theta = torch.atan2(node_vectors[..., 1], node_vectors[..., 0])
        filter_features = nn.RepresentationTensor(self.harmonics(theta), filter_rep)
        per_vector_invariants = torch.stack(
            (torch.linalg.vector_norm(node_vectors[..., :2], dim=-1), node_vectors[..., 2]),
            dim=-1,
        )
        invariant_summary = per_vector_invariants.mean(dim=-2)
        common_w_in = self.radial_in(invariant_summary)
        common_w_out = self.radial_out(invariant_summary)
        # Exact same reduced coefficients for v1, v2, v3 (option 2).
        w_in = common_w_in.unsqueeze(-2).expand(*common_w_in.shape[:-1], 3, common_w_in.shape[-1])
        w_out = common_w_out.unsqueeze(-2).expand(*common_w_out.shape[:-1], 3, common_w_out.shape[-1])

        # The same WETensorProduct is evaluated for v1, v2, v3.
        repeated_x = self.repeat_for_three_filters(features, input_rep)
        hidden_terms = self.input_layer(repeated_x, filter_features, w_in)
        h_pre = nn.RepresentationTensor(hidden_terms.tensor.sum(dim=-2), hidden_rep)
        h = self.activation(h_pre)
        repeated_h = self.repeat_for_three_filters(h, hidden_rep)
        output_terms = self.output_layer(repeated_h, filter_features, w_out)
        y = nn.RepresentationTensor(output_terms.tensor.sum(dim=-2), output_rep)
        return y, filter_features, w_in, w_out, h_pre, h

model = ExternalHarmonicNetwork().eval()
y, Y_typed, w_in, w_out, h_pre, h = model(x)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
torch.testing.assert_close(w_in[:, 0], w_in[:, 1]); torch.testing.assert_close(w_in[:, 1], w_in[:, 2])
print('one input-layer weight vector, shared over v1/v2/v3:', w_in[0, 0])
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(Y_typed)
print('sampled basis shape [batch, 3 filters, paths, out, in]:', tuple(kernel_basis.shape))

## Step 4: rotate the node and all three derived filters together

Rotating `x` automatically rotates the three vectors stored inside it. Their harmonic filters transform covariantly, while the symmetric invariant summary and its single shared coefficient vector stay fixed. No separate geometric argument is passed to the model.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    y_k, Y_k, w_in_k, w_out_k, _, _ = model(x_k)
    expected_k = y.transform_fibers(element)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=5e-5, rtol=5e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  three circular filters:', Y_k.tensor[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Step 5: follow the harmonic filter into the regular hidden state

The left heatmap contains three seven-component circular filters, one block per node vector. The right heatmap shows their summed, activated regular hidden state. Each C6 action changes every harmonic block and cyclically permutes both six-coordinate regular fields.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation = [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    y_k, Y_k, _, _, _, h_k = model(x_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(Y_k.tensor[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

flat_harmonics = harmonic_by_rotation.reshape(6, -1)
flat_labels = [f'v{vector+1}:{label}' for vector in range(3) for label in harmonic_labels]
fig_state, (ax_harm, ax_hidden) = plt.subplots(1, 2, figsize=(17, 5), constrained_layout=True)
harmonic_image = ax_harm.imshow(flat_harmonics, aspect='auto', cmap='coolwarm')
for boundary in (6.5, 13.5):
    ax_harm.axvline(boundary, color='white', linewidth=2)
ax_harm.set(xticks=range(21), xticklabels=flat_labels, yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='three shared-filter blocks', ylabel='C6 rotation', title='Y(v1) | Y(v2) | Y(v3)')
ax_harm.tick_params(axis='x', labelrotation=90, labelsize=6)
fig_state.colorbar(harmonic_image, ax=ax_harm, shrink=0.75)

image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular coordinate (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig_state.colorbar(image, ax=ax_hidden, shrink=0.75)
plt.show()

## Step 6: inspect the Wigner--Eckart kernel itself

`sample_kernel_basis` returns $K_p(v_a)$ for every vector and coupling path. One coefficient vector $w_p(I)$ is shared exactly across $a$. The actual matrix is $K_{total}=\sum_{a=1}^{3}\sum_p w_p(I)K_p(v_a)$, and the assertion verifies $h_{pre}=K_{total}x$.

In [ ]:
basis_by_vector = model.input_layer.sample_kernel_basis(Y_typed)[0].detach()
effective_kernel = torch.einsum('ap,apoi->oi', w_in[0].detach(), basis_by_vector).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=2e-5, rtol=2e-5)

fig_kernel, ax_kernel = plt.subplots(figsize=(7, 5), constrained_layout=True)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='Summed shared-filter kernel K_total')
fig_kernel.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)
plt.show()

## Step 7: unpack the final tensor-product output

The final layer produces one $xy$ vector irrep and four trivial coordinates. We combine the first trivial coordinate with $xy$ to reconstruct the physical 3D vector; the remaining three are displayed as scalar channels.

In [ ]:
fig_output = plt.figure(figsize=(12, 5), constrained_layout=True)
ax_vec = fig_output.add_subplot(1, 2, 1, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='WETensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig_output.add_subplot(1, 2, 2)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='WETensorProduct output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()